In [ ]:
ROW, COL = 3, 3

def get_partially_known_floors(known_row_0):
    floors = []

    for i in range(1 << 6):
        floor = []
        

        floor.append(list(known_row_0))
        

        for r in [1, 2]:
            row = []
            for c in range(COL):

                bit_index = (r - 1) * COL + c
                bit = (i >> bit_index) & 1
                row.append(bit)
            floor.append(row)
            
        floors.append(tuple(tuple(r) for r in floor))
    return floors


def is_goal_floor(floor):
    return all(cell == 0 for row in floor for cell in row)


def observe(floor, pos):
    """
    Kiểm tra trạng thái của 5 ô (hiện tại, trên, dưới, trái, phải)
    """
    r, c = pos
    return (
        floor[r][c],
        floor[r-1][c] if r > 0 else None,
        floor[r+1][c] if r < ROW - 1 else None,
        floor[r][c-1] if c > 0 else None,
        floor[r][c+1] if c < COL - 1 else None
    )


def transition(floor, pos, action):

    new_floor = [list(row) for row in floor]
    r, c = pos
    
    if action == "SUCK":
        new_floor[r][c] = 0
        new_pos = pos
    elif action == "UP" and r > 0:
        new_pos = (r - 1, c)
    elif action == "DOWN" and r < ROW - 1:
        new_pos = (r + 1, c)
    elif action == "LEFT" and c > 0:
        new_pos = (r, c - 1)
    elif action == "RIGHT" and c < COL - 1:
        new_pos = (r, c + 1)
    else:
        new_pos = pos # Đập tường hoặc lệnh không hợp lệ
        
    return tuple(tuple(row) for row in new_floor), new_pos


class PriorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, item):
        self.heap.append(item)
        self._upheap(len(self.heap) - 1)

    def pop(self):
        if not self.heap:
            raise IndexError("pop from empty priority queue")
        if len(self.heap) == 1:
            return self.heap.pop()
        
        root = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._downheap(0)
        return root

    def _upheap(self, index):
        parent = (index - 1) // 2
        while index > 0 and self.heap[index] < self.heap[parent]:
            self.heap[index], self.heap[parent] = self.heap[parent], self.heap[index]
            index = parent
            parent = (index - 1) // 2

    def _downheap(self, index):
        left = 2 * index + 1
        right = 2 * index + 2
        smallest = index
        
        if left < len(self.heap) and self.heap[left] < self.heap[smallest]:
            smallest = left
        if right < len(self.heap) and self.heap[right] < self.heap[smallest]:
            smallest = right
            
        if smallest != index:
            self.heap[index], self.heap[smallest] = self.heap[smallest], self.heap[index]
            self._downheap(smallest)

    def __bool__(self):
        return bool(self.heap)

    def __len__(self):
        return len(self.heap)


class BeliefNode:
    def __init__(self, states: set, parent=None, birth_action=None):
        # states là một set các tuple: ((floor), (robot_pos))
        self.states = states 
        self.parent = parent
        self.birth_action = birth_action
        self.g = 0  # Chi phí đường đi
        self.h = self.calculate_heuristic()
        self.f = self.g + self.h
        # Lưu các node con ứng với từng kết quả cảm biến: { observation: BeliefNode }
        self.children = {} 

    def calculate_heuristic(self):
        max_dirty = 0
        for floor, _ in self.states:
            dirty = sum(cell for row in floor for cell in row)
            if dirty > max_dirty:
                max_dirty = dirty
        return max_dirty

    def is_goal(self):
        """Mục tiêu đạt được khi TẤT CẢ thế giới giả thuyết đều sạch"""
        return all(is_goal_floor(floor) for floor, _ in self.states)

    def get_states(self):
        return tuple(sorted(list(self.states)))

    # Định nghĩa toán tử so sánh để dùng trong Priority Queue
    def __lt__(self, other):
        return self.f < other.f


def run_partially_observable_a_star(initial_robot_pos, known_row_0):
    possible_floors = get_partially_known_floors(known_row_0)
    
    initial_states = set((floor, initial_robot_pos) for floor in possible_floors)
    
    root = BeliefNode(states=initial_states)
    
    frontier = PriorityQueue()
    frontier.push(root)
    visited = set([root.get_states()])
    steps_explored = 0
    
    while frontier:
        current_node = frontier.pop()
        steps_explored += 1
        
        if current_node.is_goal():
            print(f"Goal Found")
            print(f"Step Explored: {steps_explored}")
            return current_node
            
        for action in ["SUCK", "UP", "DOWN", "LEFT", "RIGHT"]:
            obs_groups = {}
            for floor, pos in current_node.states:
                next_floor, next_pos = transition(floor, pos, action)
                obs = observe(next_floor, next_pos)
                
                if obs not in obs_groups:
                    obs_groups[obs] = set()
                obs_groups[obs].add((next_floor, next_pos))
            
            for obs, child_states in obs_groups.items():
                child_node = BeliefNode(
                    states=child_states, 
                    parent=current_node, 
                    birth_action=f"{action} -> (Nhìn thấy: {obs})"
                )
                child_node.g = current_node.g + 1
                child_node.f = child_node.g + child_node.h
                
                frozen = child_node.get_states()
                if frozen not in visited:
                    visited.add(frozen)
                    current_node.children[obs] = child_node
                    frontier.push(child_node)
                    
    print("Goal not found")
    return None


def main():
    """
    Tìm kiếm trong môi trường nhìn thấy 1 phần (vị trí máy, quan sát được 5 ô (hiện tại, trên, dưới, trái, phải))
    - Áp dụng A Star (Priority Queue, path cost, heuristic cost)
    - 
    """
    start_pos = (0, 0) 

    known_row_0 = (1, 0, 1) 
    
    print(f"Started position: {start_pos}")
    print(f"Know first row: {known_row_0}")
    
    result_node = run_partially_observable_a_star(start_pos, known_row_0)
    
    if result_node:
        path = []
        curr = result_node
        while curr is not None:
            path.append(curr)
            curr = curr.parent
        path.reverse()
        
        for idx, node in enumerate(path):
            if idx == 0:
                print(f"\nBước 0: Bắt đầu")
            else:
                print(f"\nBước {idx}: Thực hiện {node.birth_action}")
            
            # Lấy vị trí robot (giống nhau ở tất cả thế giới trong node này)
            rep_state = list(node.states)[0]
            robot_pos = rep_state[1]
            print(f"   - Vị trí Robot: {robot_pos}")
            print(f"   - Số lượng trạng thái sàn khả thi: {len(node.states)}")
            
if __name__ == "__main__":
    main()